In [1]:
import os, platform, requests

# força UTF-8 no Python
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

# locale pro R
os.environ["LC_ALL"] = "C.UTF-8"
os.environ["LANG"] = "C.UTF-8"

# R_HOME explícito
os.environ["R_HOME"] = r"C:\Program Files\R\R-4.5.1"

In [2]:
import pandas as pd
import numpy as np
from bcb import currency

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


In [3]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

# **CARREGAR R E BIBLIOTECAS**

In [4]:
#Apontar para a pasta certa
ro.r('.libPaths("C:/Users/FábioGaldino/AppData/Local/R/win-library/4.5")')

# Forçar locale e encoding para UTF-8
ro.r('options(encoding = "latin1")')
so = platform.system()
if so == "Windows":
    ro.r('Sys.setlocale("LC_ALL", "Portuguese_Brazil.1252")')
else:
    ro.r('Sys.setlocale("LC_ALL", "pt_BR.UTF-8")')

# Carregar pacotes
ro.r('suppressPackageStartupMessages(library(utils))')
ro.r('suppressPackageStartupMessages(library(tsibbledata))')
ro.r('suppressPackageStartupMessages(library(fpp3))')
ro.r('suppressPackageStartupMessages(library(dplyr))')
ro.r('suppressPackageStartupMessages(library(ggtime))')
ro.r('suppressPackageStartupMessages(library(patchwork))')
ro.r('suppressPackageStartupMessages(library(gridExtra))')
ro.r('suppressPackageStartupMessages(library(GGally))')
ro.r('suppressPackageStartupMessages(library(lubridate))')

%load_ext rpy2.ipython

In [5]:
%%R
options(tibble.width = Inf)
options(width = 1600)

In [6]:
def tsibble_df(name):
    with localconverter(ro.default_converter + pandas2ri.converter):
        return ro.conversion.rpy2py(ro.r(name))

In [7]:
def df_tsibble(df, index, name, key=None):
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv[name] = ro.conversion.py2rpy(df)

    ro.r(f"{name}$`{index}` <- tsibble::yearmonth({name}$`{index}`)")

    if key:
        r_cmd = f"{name} <- tsibble::as_tsibble({name}, index = `{index}`, key = {key})"
    else:
        r_cmd = f"{name} <- tsibble::as_tsibble({name}, index = `{index}`)"

    ro.r(r_cmd)

    ro.r(f"print({name})")

# **IMPORTAÇÃO**

In [8]:
precos = pd.read_excel(
    '../dados/brutos/precos_produtos.xlsx',
    dtype = {
        "IdProduto": "int64",
        "NomeMQEstat": "string",
        "GrupoIA": "string",
        "valor": "float64",
        "IdEdicao": "string",
        "Mes": "string"
    }
)

precos['Mes'] = precos['Mes'].str.strip()
precos.head()

,IdProduto,NomeMQEstat,GrupoIA,valor,IdEdicao,Mes
0,35,Aminol 806,"2,4-D AE 670",13.825234,448,2020-02
1,35,Aminol 806,"2,4-D AE 670",14.052825,446,2020-01
2,35,Aminol 806,"2,4-D AE 670",14.161839,550,2024-07
3,35,Aminol 806,"2,4-D AE 670",14.320463,548,2024-06
4,35,Aminol 806,"2,4-D AE 670",14.363840,546,2024-05


In [9]:
produtos = pd.read_excel(
    '../dados/brutos/produtos_codigos.xlsx',
    dtype = {
        "IdProduto": "int64",
        "NomeMQEstat": "string",
        "GrupoIA": "string",
        "codigo_ncm": "int64"
    }
)

produtos.head()

,IdProduto,NomeMQEstat,GrupoIA,codigo_ncm
0,35,Aminol 806,"2,4-D AE 670",29221912
1,376,U 46 BR,"2,4-D AE 670",29221912
2,475,Roundup WG,glyphosate AE 720,38089324
3,1319,Manzate 750 WG,mancozeb 750,38249986
4,2560,Zapp QI 620,glyphosate AE 500,38089324


In [10]:
cotacao_dolar = currency.get(
    'USD',
    start = '2020-01-01',
    end = '2025-12-31',
    side = 'ask'
)

cotacao_dolar = cotacao_dolar.reset_index()
cotacao_dolar.columns = ['data', 'dolar']

cotacao_dolar['ano_mes'] = cotacao_dolar['data'].dt.strftime('%Y-%m')

cotacao_dolar = cotacao_dolar.groupby('ano_mes')[['dolar']].mean().reset_index()

cotacao_dolar.head()

,ano_mes,dolar
0,2020-01,4.149464
1,2020-02,4.341011
2,2020-03,4.883855
3,2020-04,5.325580
4,2020-05,5.643445


# **IMPORTAÇÃO COMEX STAT**

In [53]:
url = "https://api-comexstat.mdic.gov.br/general"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

## **SISTEMA NCM**

Segundo a [Fazcomex](https://www.fazcomex.com.br/ncm/o-que-e-onde-consultar/), uma startup de tecnologia para comércio exterior, a NCM permite a identificação padronizada das mercadorias comercializadas no Mercosul, ou seja todo produto possui uma NCM.

Separada em várias seções, a que nos interessa é a seção VI: Produtos das Indústrias Químicas ou das Indústrias Conexas.
Nela temos dois capítulos que interessam esta análise:
- Capítulo 29 - Produtos Químicos Orgânicos;
- Capítulo 38 - Produtos diversos das indústrias químicas.

Sendo que o Capítulo 29 [inclui apenas os compostos de constituição química definida apresentados isoladamente](https://www.remessaonline.com.br/blog/ncm/capitulo/29-produtos-quimicos-organicos/). E o Capítulo 38 [pode conter tanto o ingretiente ativo técnico (38.24) ou o produjo já formulado (38.08)](https://www.remessaonline.com.br/blog/ncm/capitulo/38-produtos-diversos-das-industrias-quimicas/).

<br>

## **2,4-D AE 670: ÁCIDO DE 2,4-D 670 g/L**

Segundo a [Revista Cultivar](https://revistacultivar.com.br/fitossanidade/2-4-d), o ácido 2,4-diclorofenoxiacético, conhecido mundialmente pela sigla **2,4-D**, representa um dos herbicidas auxínicos mais importantes e amplamente utilizados na agricultura moderna.

O Brasil importa este ingrediente ativo tanto como composto de constuição química isolada (NCM 29.18.99.1.2), tanto como em produto já formulado (NCM 38.08.93.2.2):
- 29.18.99.1.2 - Ácido 2,4-diclorofenoxiacético (2,4-D), seus sais e seus ésteres;
- 38.08.93.2.2 - Outros herbicidas apresentados de outro modo, à base de ácido 2,4-diclorofenoxiacético (2,4-D), de ácido 4-(2,4-diclorofenoxi)butírico (2,4-DB), de ácido (4-cloro-2-metil)fenoxiacético (MCPA) ou de derivados de 2,4-D ou 2,4-DB.

In [84]:
payload = {
    "flow": "import",
    "monthDetail": True,
    "period": {
        "from": "2020-01",
        "to": "2025-12"
    },
    "filters": [
        {
            "filter": "ncm",
            "values": ["29189912", "38089322"]
        }
    ],
    "details": ["ncm", "country"],
    "metrics": ["metricFOB", "metricKG"]
}

response = requests.post(url, json=payload, headers=headers, verify=False)

In [86]:
importacao = pd.DataFrame(response.json()['data']['list'])
importacao.columns = ['codigo_ncm', 'ano', 'mes', 'descricao', 'pais', 'valor_usd', 'volume']
importacao['volume'] = pd.to_numeric(importacao['volume'])
importacao['valor_usd'] = pd.to_numeric(importacao['valor_usd'])
importacao['GrupoIA'] = "2,4-D AE 670"

importacao

,codigo_ncm,ano,mes,descricao,pais,valor_usd,volume,GrupoIA
0,38089322,2025,09,"Outros herbicidas apresentados de outro modo, ...",China,12296818,7903457,"2,4-D AE 670"
1,38089322,2025,11,"Outros herbicidas apresentados de outro modo, ...",China,11302287,5486426,"2,4-D AE 670"
2,38089322,2025,07,"Outros herbicidas apresentados de outro modo, ...",China,11294664,7470170,"2,4-D AE 670"
3,38089322,2025,08,"Outros herbicidas apresentados de outro modo, ...",China,9547710,6732323,"2,4-D AE 670"
4,38089322,2025,10,"Outros herbicidas apresentados de outro modo, ...",China,8660712,5362965,"2,4-D AE 670"
...,...,...,...,...,...,...,...,...
592,29189912,2020,06,"Ácido 2,4-diclorofenoxiacético (2,4-D), seus s...",Índia,36356,18050,"2,4-D AE 670"
593,38089322,2020,01,"Outros herbicidas apresentados de outro modo, ...",Índia,33706,19600,"2,4-D AE 670"
594,38089322,2020,02,"Outros herbicidas apresentados de outro modo, ...",Índia,33701,19600,"2,4-D AE 670"
595,29189912,2020,03,"Ácido 2,4-diclorofenoxiacético (2,4-D), seus s...",Suíça,36,0,"2,4-D AE 670"


## **GLYPHOSATE AE 500/720: ÁCIDO DE N-(FOSFONOMETIL)GLICINA 500/720 g/kg**

O [Glifosato](https://revistacultivar.com.br/fitossanidade/glifosato), ácido de N-(fosfonometil)glicina, possui um mecanismo bioquímico que inibe a EPSPS, atuando no controle de plantas daninhas nas lavouras agrícolas.

Também é importado como composto de constuição química isolada (NCM 29.31.49.1.4) e como produto já formulado (NCM 38.08.93.2.4):
- 29.31.49.1.4 - Glifosato e seu sal de monoisopropilamina;
- 38.08.93.2.4 - Herbicida à base de glifosato ou seus sais, de imazaquim ou de lactofen.

In [87]:
payload = {
    "flow": "import",
    "monthDetail": True,
    "period": {
        "from": "2020-01",
        "to": "2025-12"
    },
    "filters": [
        {
            "filter": "ncm",
            "values": ["29314914", "38089324"]
        }
    ],
    "details": ["ncm", "country"],
    "metrics": ["metricFOB", "metricKG"]
}

response = requests.post(url, json=payload, headers=headers, verify=False)

In [89]:
df = pd.DataFrame(response.json()['data']['list'])
df.columns = ['codigo_ncm', 'ano', 'mes', 'descricao', 'pais', 'valor_usd', 'volume']
df['volume'] = pd.to_numeric(df['volume'])
df['valor_usd'] = pd.to_numeric(df['valor_usd'])
df['GrupoIA'] = "GLYPHOSATE AE 500/720"

importacao = pd.concat([importacao, df], axis = 0, ignore_index = True)
importacao

,codigo_ncm,ano,mes,descricao,pais,valor_usd,volume,GrupoIA
0,38089322,2025,09,"Outros herbicidas apresentados de outro modo, ...",China,12296818,7903457,"2,4-D AE 670"
1,38089322,2025,11,"Outros herbicidas apresentados de outro modo, ...",China,11302287,5486426,"2,4-D AE 670"
2,38089322,2025,07,"Outros herbicidas apresentados de outro modo, ...",China,11294664,7470170,"2,4-D AE 670"
3,38089322,2025,08,"Outros herbicidas apresentados de outro modo, ...",China,9547710,6732323,"2,4-D AE 670"
4,38089322,2025,10,"Outros herbicidas apresentados de outro modo, ...",China,8660712,5362965,"2,4-D AE 670"
...,...,...,...,...,...,...,...,...
850,38089324,2020,09,"Herbicida à base de glifosato ou seus sais, de...",Taiwan (Formosa),586804,409180,GLYPHOSATE AE 500/720
851,38089324,2020,03,"Herbicida à base de glifosato ou seus sais, de...",China,328401,108000,GLYPHOSATE AE 500/720
852,38089324,2020,06,"Herbicida à base de glifosato ou seus sais, de...",Taiwan (Formosa),295726,210600,GLYPHOSATE AE 500/720
853,38089324,2020,08,"Herbicida à base de glifosato ou seus sais, de...",Taiwan (Formosa),147201,105300,GLYPHOSATE AE 500/720


## **MANCOZEB 750: MANCOZEBE 750 g/kg**

O [Mancozebe](https://revistacultivar.com.br/fitossanidade/mancozebe) é um fungicida de contato amplamente utilizado no controle preventivo de doenças fúngicas em diversas culturas agrícolas.

Sua importação se dá como ingretiente ativo técnico (NMC 38.24.99.8.6) ou o produjo já formulado (NCM 38.08.92.9.3):
- 38.24.99.8.6 - Maneb; mancozeb; cloreto de benzalcônio;
- 38.08.92.9.3 - Fungicida à base de mancozeb ou de maneb.

In [90]:
payload = {
    "flow": "import",
    "monthDetail": True,
    "period": {
        "from": "2020-01",
        "to": "2025-12"
    },
    "filters": [
        {
            "filter": "ncm",
            "values": ["38249986", "38089293"]
        }
    ],
    "details": ["ncm", "country"],
    "metrics": ["metricFOB", "metricKG"]
}

response = requests.post(url, json=payload, headers=headers, verify=False)

In [92]:
df = pd.DataFrame(response.json()['data']['list'])
df.columns = ['codigo_ncm', 'ano', 'mes', 'descricao', 'pais', 'valor_usd', 'volume']
df['volume'] = pd.to_numeric(df['volume'])
df['valor_usd'] = pd.to_numeric(df['valor_usd'])
df['GrupoIA'] = "MANCOZEB 750"

importacao = pd.concat([importacao, df], axis = 0, ignore_index = True)
importacao

,codigo_ncm,ano,mes,descricao,pais,valor_usd,volume,GrupoIA
0,38089322,2025,09,"Outros herbicidas apresentados de outro modo, ...",China,12296818,7903457,"2,4-D AE 670"
1,38089322,2025,11,"Outros herbicidas apresentados de outro modo, ...",China,11302287,5486426,"2,4-D AE 670"
2,38089322,2025,07,"Outros herbicidas apresentados de outro modo, ...",China,11294664,7470170,"2,4-D AE 670"
3,38089322,2025,08,"Outros herbicidas apresentados de outro modo, ...",China,9547710,6732323,"2,4-D AE 670"
4,38089322,2025,10,"Outros herbicidas apresentados de outro modo, ...",China,8660712,5362965,"2,4-D AE 670"
...,...,...,...,...,...,...,...,...
1332,38249986,2020,09,Maneb; mancozeb; cloreto de benzalcônio,China,911,20,MANCOZEB 750
1333,38249986,2020,03,Maneb; mancozeb; cloreto de benzalcônio,Canadá,852,0,MANCOZEB 750
1334,38249986,2020,12,Maneb; mancozeb; cloreto de benzalcônio,Índia,518,1,MANCOZEB 750
1335,38089293,2020,04,Fungicida à base de mancozeb ou de maneb,China,509,15,MANCOZEB 750


## **CLETHODIM 240: CLETODIM 240 g/L**

O [Cletodim](https://agroadvance.com.br/blog-fitotoxicidade-de-herbicidas-na-soja) é um herbicida graminicida utilizado no controle de gramíneas em culturas de folhas largas, como a soja.

No Brasil é importado como composto de constuição química isolada (NCM 29.30.90.3.9) e [como produto já formulado](https://solucoes.nortox.com.br/hc/pt-br/articles/4404082884116-Cletodim-Nortox) (NCM 38.08.93.2.9):
- 29.30.90.3.9 - Outros tioéteres, tioésteres, seus derivados e sais;
- 38.08.93.2.9 - Outros herbicidas apresentados de outro modo.

In [95]:
payload = {
    "flow": "import",
    "monthDetail": True,
    "period": {
        "from": "2020-01",
        "to": "2025-12"
    },
    "filters": [
        {
            "filter": "ncm",
            "values": ["29309039", "38089329"]
        }
    ],
    "details": ["ncm", "country"],
    "metrics": ["metricFOB", "metricKG"]
}

response = requests.post(url, json=payload, headers=headers, verify=False)

In [97]:
df = pd.DataFrame(response.json()['data']['list'])
df.columns = ['codigo_ncm', 'ano', 'mes', 'descricao', 'pais', 'valor_usd', 'volume']
df['volume'] = pd.to_numeric(df['volume'])
df['valor_usd'] = pd.to_numeric(df['valor_usd'])
df['GrupoIA'] = "CLETHODIM 240"

importacao = pd.concat([importacao, df], axis = 0, ignore_index = True)

importacao['ano_mes'] = importacao['ano'] + "-" + importacao['mes']
importacao['ano'] = pd.to_numeric(importacao['ano'])
importacao['mes'] = pd.to_numeric(importacao['mes'])

importacao

,codigo_ncm,ano,mes,descricao,pais,valor_usd,volume,GrupoIA,ano_mes
0,38089322,2025,9,"Outros herbicidas apresentados de outro modo, ...",China,12296818,7903457,"2,4-D AE 670",2025-09
1,38089322,2025,11,"Outros herbicidas apresentados de outro modo, ...",China,11302287,5486426,"2,4-D AE 670",2025-11
2,38089322,2025,7,"Outros herbicidas apresentados de outro modo, ...",China,11294664,7470170,"2,4-D AE 670",2025-07
3,38089322,2025,8,"Outros herbicidas apresentados de outro modo, ...",China,9547710,6732323,"2,4-D AE 670",2025-08
4,38089322,2025,10,"Outros herbicidas apresentados de outro modo, ...",China,8660712,5362965,"2,4-D AE 670",2025-10
...,...,...,...,...,...,...,...,...,...
2746,29309039,2020,2,"Outros tioéteres, tioésteres, seus derivados e...",França,79,0,CLETHODIM 240,2020-02
2747,29309039,2020,5,"Outros tioéteres, tioésteres, seus derivados e...",Espanha,49,0,CLETHODIM 240,2020-05
2748,29309039,2020,6,"Outros tioéteres, tioésteres, seus derivados e...",Suíça,41,0,CLETHODIM 240,2020-06
2749,29309039,2020,11,"Outros tioéteres, tioésteres, seus derivados e...",França,4,0,CLETHODIM 240,2020-11


## **TRATAMENTO DE DADOS**

In [100]:
total_por_ing_atv = importacao.groupby('GrupoIA').agg(
    num_registros_total = ('valor_usd', 'count'),
    valor_usd_total = ('valor_usd', 'sum'),
    vol_total = ('volume', 'sum')
).reset_index()

vol_zero = importacao[importacao['volume'] == 0].groupby('GrupoIA').agg(
    num_registros_vol_zero = ('valor_usd', 'count'),
    valor_usd_vol_zero = ('valor_usd', 'sum')
).reset_index()

vol_zero = total_por_ing_atv.merge(vol_zero, on = 'GrupoIA', how = 'left').fillna(0)

vol_zero['pct_vol_zero'] = np.round(vol_zero['num_registros_vol_zero'] / vol_zero['num_registros_total'] * 100, 2)
vol_zero['pct_valor_usd'] = np.round(vol_zero['valor_usd_vol_zero'] / vol_zero['valor_usd_total'] * 100, 5)

vol_zero

,GrupoIA,num_registros_total,valor_usd_total,vol_total,num_registros_vol_zero,valor_usd_vol_zero,pct_vol_zero,pct_valor_usd
0,"2,4-D AE 670",597,992832462,420393414,42,14183,7.04,0.00143
1,CLETHODIM 240,1414,7256804091,946833719,127,43557,8.98,0.00060
2,GLYPHOSATE AE 500/720,258,4928639563,997779550,23,3798,8.91,0.00008
3,MANCOZEB 750,482,1688515476,461079595,12,2870,2.49,0.00017


Foram identificados 204 registros (7,4% do total) com volume zerado (metricKG = 0), porém com valor FOB registrado. A proporção varia por ingrediente ativo: Clethodim (8,98%), Glifosato (8,91%), 2,4-D (7,04%) e Mancozeb (2,49%). Em todos os casos, estes registros representam menos de 0,002% do valor total importado por grupo, provavelmente correspondendo a amostras comerciais ou remessas para homologação regulatória. Para o cálculo de preço unitário (USD/kg), estes registros foram excluídos da análise.

In [102]:
importacao = importacao[importacao['volume'] > 0].copy().reset_index(drop = True)
importacao

,codigo_ncm,ano,mes,descricao,pais,valor_usd,volume,GrupoIA,ano_mes
0,38089322,2025,9,"Outros herbicidas apresentados de outro modo, ...",China,12296818,7903457,"2,4-D AE 670",2025-09
1,38089322,2025,11,"Outros herbicidas apresentados de outro modo, ...",China,11302287,5486426,"2,4-D AE 670",2025-11
2,38089322,2025,7,"Outros herbicidas apresentados de outro modo, ...",China,11294664,7470170,"2,4-D AE 670",2025-07
3,38089322,2025,8,"Outros herbicidas apresentados de outro modo, ...",China,9547710,6732323,"2,4-D AE 670",2025-08
4,38089322,2025,10,"Outros herbicidas apresentados de outro modo, ...",China,8660712,5362965,"2,4-D AE 670",2025-10
...,...,...,...,...,...,...,...,...,...
2542,29309039,2020,6,"Outros tioéteres, tioésteres, seus derivados e...",Dinamarca,1363,100,CLETHODIM 240,2020-06
2543,29309039,2020,4,"Outros tioéteres, tioésteres, seus derivados e...",Taiwan (Formosa),805,200,CLETHODIM 240,2020-04
2544,29309039,2020,8,"Outros tioéteres, tioésteres, seus derivados e...",Estados Unidos,691,3,CLETHODIM 240,2020-08
2545,29309039,2020,3,"Outros tioéteres, tioésteres, seus derivados e...",França,569,1,CLETHODIM 240,2020-03


## **PREÇO IMPORTAÇÃO (R$)**

In [115]:
importacao = importacao.merge(cotacao_dolar, on = 'ano_mes', how = 'left')
importacao['preco_import'] = (importacao['valor_usd'] / importacao['volume']) * importacao['dolar']
importacao = importacao.drop('dolar', axis = 1)
importacao

,codigo_ncm,ano,mes,descricao,pais,valor_usd,volume,GrupoIA,ano_mes,preco_import
0,38089322,2025,9,"Outros herbicidas apresentados de outro modo, ...",China,12296818,7903457,"2,4-D AE 670",2025-09,8.351036
1,38089322,2025,11,"Outros herbicidas apresentados de outro modo, ...",China,11302287,5486426,"2,4-D AE 670",2025-11,11.002399
2,38089322,2025,7,"Outros herbicidas apresentados de outro modo, ...",China,11294664,7470170,"2,4-D AE 670",2025-07,8.358906
3,38089322,2025,8,"Outros herbicidas apresentados de outro modo, ...",China,9547710,6732323,"2,4-D AE 670",2025-08,7.724770
4,38089322,2025,10,"Outros herbicidas apresentados de outro modo, ...",China,8660712,5362965,"2,4-D AE 670",2025-10,8.697146
...,...,...,...,...,...,...,...,...,...,...
2542,29309039,2020,6,"Outros tioéteres, tioésteres, seus derivados e...",Dinamarca,1363,100,CLETHODIM 240,2020-06,70.829658
2543,29309039,2020,4,"Outros tioéteres, tioésteres, seus derivados e...",Taiwan (Formosa),805,200,CLETHODIM 240,2020-04,21.435460
2544,29309039,2020,8,"Outros tioéteres, tioésteres, seus derivados e...",Estados Unidos,691,3,CLETHODIM 240,2020-08,1257.904078
2545,29309039,2020,3,"Outros tioéteres, tioésteres, seus derivados e...",França,569,1,CLETHODIM 240,2020-03,2778.913236


## **PRÉ-PROCESSAMENTO**

In [132]:
importacao_backup = importacao.copy()

### **VOLUME IMPORTADO**

In [135]:
importacao.groupby(['GrupoIA', 'ano'])['volume'].agg(
    minimo = 'min',
    media = 'mean',
    maximo = 'max',
    std = 'std',
    q25 = lambda x: x.quantile(0.25),
    q50 = lambda x: x.quantile(0.50),
    q75 = lambda x: x.quantile(0.75)
)

minimo         media    maximo           std  \
GrupoIA               ano                                                  
2,4-D AE 670          2020   18050  5.267290e+05   2857600  5.354681e+05   
                      2021   17600  5.372225e+05   2873201  5.208278e+05   
                      2022       1  7.353233e+05   6017877  1.047706e+06   
                      2023    1714  9.226409e+05   5826068  1.255305e+06   
                      2024      64  8.357089e+05   5679998  1.109529e+06   
                      2025      20  1.214258e+06   7903457  1.701876e+06   
CLETHODIM 240         2020       1  2.873542e+05   5242225  6.364428e+05   
                      2021       1  4.448843e+05   7942396  1.231025e+06   
                      2022       1  8.341409e+05  22116611  2.925083e+06   
                      2023       1  6.766949e+05  22654865  2.898933e+06   
                      2024       1  9.079650e+05  30716837  3.946617e+06   
                      2025       1  1.197596e+06  47457451  5.480027e+06   
GLYPHOSATE AE 500/720 2020   51160  3.379174e+06  11947979  3.979232e+06   
                      2021   42120  2.253887e+06  10928861  3.301850e+06   
                      2022       8  4.535293e+06  22791097  6.081725e+06   
                      2023      48  3.070031e+06  22063373  4.692836e+06   
                      2024      24  5.368405e+06  28884704  6.608462e+06   
                      2025       1  5.549655e+06  26820554  7.321400e+06   
MANCOZEB 750          2020       1  6.409517e+05   4263200  1.120921e+06   
                      2021       5  8.462786e+05   4432320  1.275078e+06   
                      2022       1  8.643074e+05   5623900  1.257544e+06   
                      2023       1  7.544876e+05   6891460  1.421035e+06   
                      2024       5  1.278557e+06  10203950  2.074419e+06   
                      2025       1  1.505978e+06  10142000  2.262255e+06   

                                   q25        q50         q75  
GrupoIA               ano                                      
2,4-D AE 670          2020   164060.00   380040.0   687600.00  
                      2021   180000.00   381576.0   696600.00  
                      2022   154880.00   329120.0   932832.00  
                      2023   268650.00   535080.0   928000.00  
                      2024   165115.00   411600.0  1095102.00  
                      2025   249000.00   590379.0  1247975.00  
CLETHODIM 240         2020     7835.00    67471.0   297635.50  
                      2021     9134.25    60123.0   279969.25  
                      2022    14898.00    85790.0   364708.00  
                      2023    10133.00    68150.0   267190.00  
                      2024     7714.50    42636.0   171721.50  
                      2025     1905.00    59799.5   205192.75  
GLYPHOSATE AE 500/720 2020   409180.00  1144800.0  7533027.00  
                      2021   187161.00   487879.5  2469960.00  
                      2022   275464.75  1561216.0  6518994.75  
                      2023   379800.00  1561100.0  2770000.00  
                      2024  1334625.00  2820000.0  6824850.00  
                      2025  1132751.50  2577000.0  5924120.25  
MANCOZEB 750          2020     1658.00    26980.0   645037.50  
                      2021     2598.00    38015.0  1424427.50  
                      2022      986.00    50000.0  1467280.00  
                      2023     1572.00    80620.0   743165.00  
                      2024    24550.00   329027.5  1447246.25  
                      2025    43700.00   659880.5  1681596.00

In [139]:
importacao = importacao[importacao['volume'] >= 986].copy().reset_index()

importacao.groupby(['GrupoIA', 'ano'])['volume'].agg(
    minimo = 'min',
    media = 'mean',
    maximo = 'max',
    std = 'std',
    q25 = lambda x: x.quantile(0.25),
    q50 = lambda x: x.quantile(0.50),
    q75 = lambda x: x.quantile(0.75)
)

minimo         media    maximo           std  \
GrupoIA               ano                                                  
2,4-D AE 670          2020   18050  5.267290e+05   2857600  5.354681e+05   
                      2021   17600  5.372225e+05   2873201  5.208278e+05   
                      2022    1097  7.421318e+05   6017877  1.050165e+06   
                      2023    1714  9.226409e+05   5826068  1.255305e+06   
                      2024   16720  8.582916e+05   5679998  1.115857e+06   
                      2025   45504  1.248461e+06   7903457  1.713425e+06   
CLETHODIM 240         2020     996  3.395655e+05   5242225  6.792297e+05   
                      2021    1089  5.413352e+05   7942396  1.339197e+06   
                      2022    1338  9.837037e+05  22116611  3.154540e+06   
                      2023    1000  8.158631e+05  22654865  3.166759e+06   
                      2024    1156  1.139837e+06  30716837  4.394364e+06   
                      2025    1500  1.586948e+06  47457451  6.263698e+06   
GLYPHOSATE AE 500/720 2020   51160  3.379174e+06  11947979  3.979232e+06   
                      2021   42120  2.253887e+06  10928861  3.301850e+06   
                      2022   16704  4.703266e+06  22791097  6.130107e+06   
                      2023   22464  3.203493e+06  22063373  4.750211e+06   
                      2024   40000  5.493251e+06  28884704  6.633961e+06   
                      2025   23400  5.955727e+06  26820554  7.425915e+06   
MANCOZEB 750          2020    1440  8.359601e+05   4263200  1.216188e+06   
                      2021    2316  1.112628e+06   4432320  1.358470e+06   
                      2022    1544  1.162819e+06   5623900  1.335799e+06   
                      2023    1544  9.927037e+05   6891460  1.558122e+06   
                      2024    1544  1.482347e+06  10203950  2.166297e+06   
                      2025   19000  1.771708e+06  10142000  2.357254e+06   

                                   q25        q50         q75  
GrupoIA               ano                                      
2,4-D AE 670          2020   164060.00   380040.0   687600.00  
                      2021   180000.00   381576.0   696600.00  
                      2022   156845.00   331760.0   933624.00  
                      2023   268650.00   535080.0   928000.00  
                      2024   199678.00   414025.5  1135968.00  
                      2025   256664.00   600267.0  1258330.50  
CLETHODIM 240         2020    30046.50   137150.0   356235.75  
                      2021    28728.00   130416.0   338650.00  
                      2022    29515.00   111964.5   440537.50  
                      2023    38943.50   111024.0   321076.00  
                      2024    25709.25    82756.5   235412.50  
                      2025    35030.00   135313.0   296230.00  
GLYPHOSATE AE 500/720 2020   409180.00  1144800.0  7533027.00  
                      2021   187161.00   487879.5  2469960.00  
                      2022   329911.50  1952216.0  7188998.25  
                      2023   463354.00  1700000.0  2790000.00  
                      2024  1561750.00  2880000.0  7017500.00  
                      2025  1320000.00  2584000.0  6709101.00  
MANCOZEB 750          2020    16800.00    96000.0  1453200.00  
                      2021    19000.00   425680.0  2146470.00  
                      2022    25700.00   858640.0  1758964.00  
                      2023    52800.00   240000.0  1151350.00  
                      2024    83650.00   528000.0  1749401.00  
                      2025   133800.25   886697.0  2410802.75

In [126]:
importacao.groupby(['GrupoIA', 'ano'])['preco_import'].agg(
    minimo = 'min',
    media = 'mean',
    maximo = 'max',
    std = 'std',
    q25 = lambda x: x.quantile(0.25),
    q50 = lambda x: x.quantile(0.50),
    q75 = lambda x: x.quantile(0.75)
)

minimo       media        maximo          std  \
GrupoIA               ano                                                      
2,4-D AE 670          2020   3.843536   10.988604     15.429098     2.245751   
                      2021   6.799688   12.995968     26.573943     2.883794   
                      2022   8.938238   17.290652     37.322202     4.678996   
                      2023   6.962594   14.861758     37.024955     6.925790   
                      2024   6.768184   22.389683    947.521313   107.570390   
                      2025   7.521804   77.684117   2468.439475   396.369719   
CLETHODIM 240         2020   8.258638  252.767486   5074.344261   676.152938   
                      2021   8.954369  177.836815   2739.531980   332.573211   
                      2022   8.119767  175.062659   2711.711333   312.370788   
                      2023   5.362820  282.196664   5371.114667   626.392277   
                      2024   9.928788  402.533580  12659.482691  1174.479990   
                      2025   9.532696  580.254110  16614.347978  1914.065967   
GLYPHOSATE AE 500/720 2020   6.923107   13.989065     18.821391     4.582212   
                      2021  10.325082   23.657897     46.787621     9.903846   
                      2022   1.118106   37.471408     63.125104    15.375897   
                      2023   1.021572   26.412233     56.339870    13.394505   
                      2024   0.925436   19.948149     33.105347     5.933945   
                      2025   8.211677  121.883906   4089.965727   616.536836   
MANCOZEB 750          2020   7.017780  777.297349  15143.412060  2478.183036   
                      2021  10.312960  943.837803  16857.718758  3065.541085   
                      2022   0.873739  750.464682  10627.694785  2211.456474   
                      2023   0.786423  761.541744  15094.091191  2553.040701   
                      2024   0.898162  755.677388  17716.169068  3027.773854   
                      2025   9.300142  881.447231  18494.304493  3130.927806   

                                  q25        q50         q75  
GrupoIA               ano                                     
2,4-D AE 670          2020   9.779908  11.185139   12.510606  
                      2021  11.232466  13.037441   14.473227  
                      2022  13.961596  16.702738   20.373189  
                      2023   9.723610  12.599573   18.036539  
                      2024   8.267216   9.541031   10.965069  
                      2025   9.772827  10.714212   12.116546  
CLETHODIM 240         2020  34.977046  59.636035  141.134867  
                      2021  31.490511  70.029407  129.122194  
                      2022  36.938614  68.175431  162.362575  
                      2023  34.439561  73.954716  240.078650  
                      2024  31.802940  62.842683  411.340392  
                      2025  33.416615  87.030030  417.316087  
GLYPHOSATE AE 500/720 2020   7.743389  16.415978   17.735224  
                      2021  16.575922  19.316091   27.404933  
                      2022  28.248963  38.079494   48.491126  
                      2023  18.167542  23.608834   31.265788  
                      2024  16.960743  19.408129   21.210010  
                      2025  15.995202  19.363676   21.254131  
MANCOZEB 750          2020  12.603345  16.980814   26.345380  
                      2021  12.816291  17.941369   23.185063  
                      2022  16.172306  20.170248   42.754158  
                      2023  13.488580  17.661530   38.672462  
                      2024  11.958953  15.238786   22.366174  
                      2025  14.561737  16.987903   28.036380

### **2,4-D AE 670**

In [131]:
importacao[(importacao['GrupoIA'] == "2,4-D AE 670") & (importacao['preco_import'] > 50)]

,codigo_ncm,ano,mes,descricao,pais,valor_usd,volume,GrupoIA,ano_mes,preco_import
71,29189912,2025,6,"Ácido 2,4-diclorofenoxiacético (2,4-D), seus s...",Japão,13350,30,"2,4-D AE 670",2025-06,2468.439475
72,29189912,2025,11,"Ácido 2,4-diclorofenoxiacético (2,4-D), seus s...",Japão,8900,20,"2,4-D AE 670",2025-11,2376.679421
147,38089322,2024,9,"Outros herbicidas apresentados de outro modo, ...",Taiwan (Formosa),10943,64,"2,4-D AE 670",2024-09,947.521313


# **ANÁLISE**

## **RANKING DAS MOLÉCULAS POR VOLUME**

In [ ]:
ranking = importacao.groupby(['ano', 'codigo_ncm']).agg({
    'volume': 'sum',
    'valor_usd': 'sum',
    'pais': 'nunique'
}).reset_index().sort_values(by = ['ano', 'volume'], ascending = [False, False]).reset_index(drop = True)

ranking.columns = ['ano', 'codigo_ncm', 'volume', 'valor_usd', 'qtde_paises']
ranking = ranking.merge(produtos[['codigo_ncm', 'GrupoIA']].drop_duplicates(subset = 'codigo_ncm'), on = 'codigo_ncm', how = 'left')
ranking = ranking[['ano', 'codigo_ncm', 'GrupoIA', 'volume', 'valor_usd', 'qtde_paises']]

ranking.to_excel('../dados/processados/ranking.xlsx')
ranking

## **TENDÊNCIA DE PREÇOS**

In [ ]:
importacao.groupby(['ano_mes', 'codigo_ncm'])[['preco_importacao']].mean().reset_index()

In [ ]:
precosdf = precos.copy()
precosdf = precosdf.groupby(['Mes', 'IdProduto']).agg({
    'valor': 'mean',
    'NomeMQEstat': 'first'
}).reset_index()
precosdf = precosdf.merge(produtos[['IdProduto', 'codigo_ncm']], on = 'IdProduto', how = 'left')
precosdf['key'] = precosdf['codigo_ncm'].astype(str) + "-" + precosdf['Mes']

imp = importacao.groupby(['ano_mes', 'codigo_ncm'])[['preco_importacao']].mean().reset_index()
imp['key'] = imp['codigo_ncm'].astype(str) + "-" + imp['ano_mes']

precosdf = precosdf.merge(imp[['key', 'preco_importacao']], on = 'key', how = 'left')
precosdf = precosdf.drop('key', axis = 1)

precosdf.head()

In [ ]:
precosdf[['IdProduto', 'NomeMQEstat']].drop_duplicates('IdProduto')

### **PRODUTO: AMINOL 806 (ID 35)**

In [ ]:
preco_35 = precosdf[precosdf['IdProduto'] == 35].copy().reset_index(drop = True)
preco_35['ano_mes'] = pd.to_datetime(preco_35["Mes"] + "-01")
#preco_35 = preco_35.groupby('ano_mes')[['valor']].mean().reset_index()
preco_35.head()

In [ ]:
df_tsibble(preco_35, 'ano_mes', 'preco_35')

In [ ]:
%%R -w 900 -h 400 -u px
preco_35 |> 
    autoplot()

In [ ]:
%%R -w 1200 -h 800 -u px
preco_35 |>
    model(stl = STL(valor)) -> dcmp_preco_35

components(dcmp_preco_35) |> autoplot()

### **PRODUTO: POQUER (ID 2734)**

In [ ]:
preco_2734 = precosdf[precosdf['IdProduto'] == 2734].copy().reset_index(drop = True)
preco_2734['ano_mes'] = pd.to_datetime(preco_2734["Mes"] + "-01")
preco_2734 = preco_2734[['ano_mes', 'valor', 'preco_importacao']].copy()
preco_2734